# M10 · Seismograms: reading, filtering, spectra, record sections

**Week 2 · about 9.7 h (+ 1.7 h optional)** · before this: M01, M02, M03

> **한국어 요약** M03에서 받은 2019년 상주 지진(ML 3.9)의 파형을 ObsPy로 읽고, SAC 헤더의 시각을 UTC로 바꿉니다. 필터와 진폭 스펙트럼, 계기 응답 제거를 익히고, 공유 서버에서 스레드 수를 제한하는 방법을 직접 재어 봅니다. S-P 시간으로 거리를 구하고 레코드 섹션을 그립니다.

**You will**

- Turn SAC header times into UTC moments
- Filter a seismogram and compute its amplitude spectrum
- Turn counts into ground velocity and test the result
- Estimate distances from S-P times and draw a record section

**New words:** trace (ObsPy Trace; 트레이스), reference time (SAC; 기준 시각), Nyquist frequency (나이퀴스트 주파수), aliasing (에일리어싱(겹침 현상)), zero-phase filter (영위상 필터), spectrum (스펙트럼), S-P time (S-P 시간), record section (레코드 섹션(거리별 파형 배열도)) · [all terms](../../../docs/glossary.md)

**Optional reading:** Helffrich, Wookey & Bastow (2013), SAC files and headers; Scherbaum (2001), causal and zero-phase filters

Log any AI help in `work/ai_log/M10.md` ([rules](../../../docs/ai_policy.md)); pass criteria: [criteria/M10.yaml](../../../.sgtl/criteria/M10.yaml).

---

© 2026 Minseong Seo (SGTL, Seoul National University) and contributors · code MIT · text CC BY 4.0 · cite CITATION.cff

In [ ]:
# Parameters: the only cell with values you may change. Run it first, then the setup cell.
DATASET = "sangju2019"                    # your STP download of M03 (a folder in the data folder)
KMA_EVENT_ID = "2019009762"               # the ML 3.9 Sangju earthquake of 21 July 2019 (KMA catalog)
BAND = "HH"                               # HH (broadband), HG (accelerometer) or EL (short period)
NEAR_STATION = "KS.SAJB"                  # the station nearest to the epicenter
FMIN_HZ, FMAX_HZ = 1.0, 10.0              # band-pass corner frequencies (Hz)
SCALE_KM = 2.0                            # record section: half-height of each trace (km)
VELOCITY_MODEL = "kim1983"                # 1-D model; its top layer gives VP and VS (Part 7)
PRE_FILT_HZ = (0.3, 0.6, 30.0, 40.0)      # response removal: pre-filter corner frequencies (Hz), as in M03
N_COLOCATED = 6                           # Part 6: nearest stations with both HH and HG
N_PICKED = 8                              # Part 7: nearest stations for S-P distances
PGV_TRACE = "KS.CHJ2..HHE"                # stretch task: the HH horizontal channel you correct
FPEAK_BAND_HZ = (1.0, 20.0)               # stretch task: band of the spectral peak (Hz)
FIGURE_DIR = "figures"                    # your figures go here, next to this notebook
RESULT_DIR = "results"                    # your result files go here, next to this notebook

In [ ]:
import sgtl_tutorial as st
S = st.init("M10")

import json
import os
import shutil
import subprocess
import sys
import textwrap
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from obspy import Stream, Trace, UTCDateTime, read, read_inventory
from obspy.geodetics import gps2dist_azimuth

Path(FIGURE_DIR).mkdir(exist_ok=True)
Path(RESULT_DIR).mkdir(exist_ok=True)

## Part 0 · Your data (about 15 min)

The waveforms come from your STP download of M03. The cell finds the ML 3.9 earthquake in the KMA catalog (UTC)
and its event folder. It also reads the `kim1983` **velocity model** (P and S speeds with depth) for Part 7.

In [ ]:
cat = st.data.kma_catalog()
event = cat.loc[cat["event_id"] == KMA_EVENT_ID].iloc[0]
ORIGIN = UTCDateTime(event["origin_utc"].isoformat())
EV_LAT, EV_LON = float(event["latitude"]), float(event["longitude"])
EV_DEPTH_KM, EV_ML = float(event["depth_km"]), float(event["ml"])
print(f"KMA event {KMA_EVENT_ID}: origin {ORIGIN} (UTC), {EV_LAT:.2f} N, {EV_LON:.2f} E, depth {EV_DEPTH_KM:g} km, "
      f"ML {EV_ML:.1f}")

DATA_DIR = st.data.root() / DATASET
SAC_DIR = DATA_DIR / "stp_download" / "SAC" / ORIGIN.strftime("%Y%m%d%H%M%S")
META_DIR = st.data.root() / "metadata"
if not SAC_DIR.is_dir():
    raise FileNotFoundError(f"There are no waveforms of this event in {SAC_DIR}. Fetch the {DATASET} data with STP "
                            "first (module M03), or set DATASET in the parameters cell to the folder you used.")
for band_dir in sorted(p for p in SAC_DIR.iterdir() if p.is_dir()):
    files = sorted(band_dir.glob("*.sac"))
    stations = {".".join(f.name.split(".")[1:3]) for f in files}
    print(f"{band_dir.name}: {len(files):4d} SAC files from {len(stations):3d} stations")

model = st.data.velocity_model(VELOCITY_MODEL)
VP_KM_S, VS_KM_S = float(model.iloc[0, 1]), float(model.iloc[0, 2])
print(f"{VELOCITY_MODEL}, top layer ({model.columns[1]}, {model.columns[2]}): VP {VP_KM_S} km/s, VS {VS_KM_S} km/s;",
      model.attrs["citation"].split(".")[0])

Each record runs from 30 s before to 470 s after the origin.

**SeisComP:** view this event in `scrttv` too ([workshop](../../../docs/seiscomp_gui.md)) and compare raw Z, N and
E. Its causal filter will not match the zero-phase filter here.

## Part 1 · Seismograms in ObsPy: Stream and Trace (about 40 min)

ObsPy keeps one channel's record in a **Trace**: the numbers in `tr.data` and their description (station,
channel, start time, sampling rate) in `tr.stats`. A **Stream** is a list of Traces; `obspy.read` makes one from
files. The numbers are **counts**, not yet m/s, and each channel has an offset. The nearest station:

In [ ]:
near = read(str(SAC_DIR / BAND / f"*.{NEAR_STATION}.{BAND}?.sac"))
near.sort(keys=["channel"])
print(near)
z_raw = near.select(component="Z")[0]
print(z_raw.stats)
print("first five samples:", z_raw.data[:5], "| type of the numbers:", z_raw.data.dtype)

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(7.5, 5.2), sharex=True)
for ax, trace in zip(axes, near):
    ax.plot(trace.times(), trace.data, color="k", lw=0.4)
    ax.set_ylabel(f"{trace.stats.channel} (counts)")
axes[0].set_title(f"{NEAR_STATION}, raw records", fontsize=10)
axes[-1].set_xlabel("Time after the first sample (s)")
fig.tight_layout()
fig.savefig(Path(FIGURE_DIR) / "near_station_raw.png", dpi=150)
for trace in near:
    print(f"{trace.id}: {trace.stats.npts} samples at {trace.stats.sampling_rate:g} Hz, mean {trace.data.mean():.0f}, "
          f"largest |value| {np.abs(trace.data).max()} counts")

## Part 2 · SAC files: headers and time (about 40 min)

A SAC file holds a **header**, then the samples. Every SAC time counts seconds from one **reference time**:

- `nzyear`, `nzjday`, `nzhour`, `nzmin`, `nzsec`, `nzmsec`: the reference time in UTC (`nzjday` is the day of the
  year; `nzmsec` the milliseconds);
- `b`, `e`: the first and last sample; `delta`: seconds between samples; `npts`: the number of samples;
- `o`: the origin time; `a`, `t0`: **phase picks**, measured arrival times (here P in `a`, S in `t0`).

A header that is not set holds −12345; ObsPy leaves it out of `tr.stats.sac`.

In [ ]:
head = z_raw.stats.sac
for key in ["nzyear", "nzjday", "nzhour", "nzmin", "nzsec", "nzmsec", "b", "e", "delta", "npts", "o", "a", "t0",
            "stla", "stlo", "stel", "evla", "evlo", "evdp", "dist", "az", "baz", "knetwk", "kstnm", "kcmpnm"]:
    print(f"{key:>7s} = {head.get(key, '(not set)')}")
print("day of the year", head.nzjday, "of", head.nzyear, "is", UTCDateTime(year=head.nzyear, julday=head.nzjday).date)
print("ObsPy's start time (first sample):", z_raw.stats.starttime)
print("KMA origin time:                  ", ORIGIN)

The picks are empty: M20 adds them. `b` is not 0: ObsPy's start time is the reference time plus `b`.

**Watch out:** `o` says 0.0, but the records start 30 s before the KMA origin time. Here the reference time is the
start of the requested window, not the origin. STP also wrote its event list:

In [ ]:
kst_rows = pd.read_csv(DATA_DIR / "event_catalog" / "event_catalog.csv")
print(kst_rows.head(3).to_string(index=False))

**Watch out:** these rows are in KST (UTC + 9 h), unlike SAC files: 11:04:18 KST is 02:04:18 UTC. Convert to UTC
once and stay there.

### M10-T1 · Pick times from a SAC header · core · about 60 min

Write `pick_times_utc(sac_path)`. It returns `(p_utc, s_utc)`: the P pick (header `a`) and the S pick (header
`t0`) as ObsPy `UTCDateTime` moments. A pick that is not set gives `None`; your own files have no picks yet.

Example (a made-up file):

- reference time 2030-05-06 07:08:09.000 (`nzjday` 126), `b` = 0.0084, `o` = 30.0, `a` = 41.96, no `t0`;
- result: `(UTCDateTime("2030-05-06T07:08:50.960"), None)`.

In [ ]:
# Before you code: in the example of the task, what would you get if you added a to ObsPy's start time instead?
st.predict("M10-T1", """

""")

In [ ]:
def pick_times_utc(sac_path):
    """UTC moments of the P pick (SAC header a) and the S pick (header t0) of one SAC file.

    sac_path : str or pathlib.Path of a SAC file
    returns  : (p_utc, s_utc), each an obspy UTCDateTime, or None when that pick is not set in the header.
               All SAC times (b, e, o, a, t0) are seconds after the reference time: nzyear, nzjday (the day of
               the year), nzhour, nzmin, nzsec and nzmsec (milliseconds), in UTC.
    """
    # Step 1: read the file and find the reference time (b, the first sample, and the nz* numbers are linked).
    # Step 2: for each of the headers a and t0: None when it is missing, else the reference time plus its seconds.
    # YOUR CODE HERE
    raise NotImplementedError("M10-T1: write your code here")

In [ ]:
st.check("M10-T1", func=pick_times_utc, dataset=DATASET)

In [ ]:
near_z_file = sorted((SAC_DIR / BAND).glob(f"*.{NEAR_STATION}.{BAND}Z.sac"))[0]
print(f"your function on {near_z_file.name}:", pick_times_utc(near_z_file))

**Explain:** What would go wrong in a location (M31) if you took `o = 0.0` of these files at face value?

## Part 3 · Sampling, the Nyquist frequency and aliasing (about 30 min)

A digitizer samples the ground motion at a **sampling rate**, here 100 samples per second. A record cannot hold
frequencies above the **Nyquist frequency**, half that rate: a wave above it shows up at a false, lower frequency
(**aliasing**). The cell samples a 30 Hz sine at 100 Hz, then keeps every second sample, once without a filter and
once with `decimate`.

In [ ]:
rate_hz, sine_hz = 100.0, 30.0
t_sine = np.arange(0.0, 4.0, 1.0 / rate_hz)
sine = Trace(np.sin(2.0 * np.pi * sine_hz * t_sine), header={"sampling_rate": rate_hz})
kept_plain = sine.copy().decimate(2, no_filter=True)     # every second sample, nothing else
kept_filtered = sine.copy().decimate(2)                  # ObsPy low-pass filters before it drops samples


def zero_crossing_hz(trace):
    """Frequency estimated from the number of sign changes per second (two per cycle)."""
    signs = np.signbit(trace.data)
    return np.count_nonzero(signs[1:] != signs[:-1]) / (trace.stats.npts * trace.stats.delta) / 2.0


for label, trace in (("100 Hz samples", sine), ("50 Hz, no filter", kept_plain), ("50 Hz, decimate", kept_filtered)):
    largest = np.abs(trace.data[20:-20]).max()           # leave out the first and last 20 samples (filter edges)
    seen = f"{zero_crossing_hz(trace):5.1f} Hz" if largest > 0.1 else "none (almost nothing left)"
    print(f"{label:>17s}: Nyquist {trace.stats.sampling_rate / 2:4.0f} Hz, largest |value| {largest:.3f}, "
          f"apparent frequency {seen}")

fig, ax = plt.subplots(figsize=(7.5, 2.6))
ax.plot(sine.times(), sine.data, ".-", color="0.6", lw=0.6, ms=3, label="30 Hz sine, sampled at 100 Hz")
ax.plot(kept_plain.times(), kept_plain.data, "s-", color="tab:red", lw=0.8, ms=3, label="every second sample")
ax.set_xlim(0.0, 0.3)
ax.set_xlabel("Time (s)")
ax.set_ylabel("Amplitude")
fig.legend(loc="upper center", ncol=2, frameon=False, fontsize=8)
fig.tight_layout(rect=(0, 0, 1, 0.84))
fig.savefig(Path(FIGURE_DIR) / "aliasing.png", dpi=150)

The red samples trace a 20 Hz wave that was never there (50 − 30 = 20). `decimate` filtered first and removed the
30 Hz wave (largest value 0.005).

**Watch out:** filter before you keep fewer samples, or false waves appear.

## Part 4 · Detrend, taper, filter: zero-phase versus causal (about 60 min)

Before filtering, **detrend** a record (remove its mean or a straight line) and **taper** it (fade both ends to
zero). A **band-pass filter** keeps the frequencies between two **corner frequencies**. This module uses a
**Butterworth** filter. ObsPy's `corners` argument is the filter order, not a corner frequency: more corners
cut more steeply.

Filters come in two kinds:

- A **causal filter** uses only past samples, like an instrument: the wave comes out later and changed in shape.
- A **zero-phase filter** runs it forward and backward (`zerophase=True`): peaks stay in place, but a sharp onset
  grows small wiggles ahead of itself.

The cell filters a made-up signal that starts sharply at 8 s.

In [ ]:
t_demo = np.arange(0.0, 20.0, 0.01)
onset_s = 8.0
burst = np.where(t_demo >= onset_s, np.exp(-(t_demo - onset_s) / 0.4) * np.sin(2.0 * np.pi * 4.0 * (t_demo - onset_s)),
                 0.0)
plain = Trace(burst, header={"delta": 0.01})
versions = {
    "unfiltered": plain,
    "causal": plain.copy().filter("bandpass", freqmin=FMIN_HZ, freqmax=FMAX_HZ, corners=4, zerophase=False),
    "zero-phase": plain.copy().filter("bandpass", freqmin=FMIN_HZ, freqmax=FMAX_HZ, corners=4, zerophase=True),
}
fig, axes = plt.subplots(3, 1, figsize=(7.5, 5.0), sharex=True)
for ax, (name, version) in zip(axes, versions.items()):
    size = np.abs(version.data)
    first = t_demo[np.argmax(size > 0.01 * size.max())]
    peak = t_demo[np.argmax(size)]
    print(f"{name:>10s}: first above 1 % of its peak at {first:.2f} s; peak at {peak:.2f} s")
    ax.plot(t_demo, version.data, color="k", lw=0.8)
    ax.axvline(onset_s, color="tab:red", lw=0.8, ls="--")
    ax.set_ylabel(name)
axes[-1].set_xlim(6.5, 10.0)
axes[-1].set_xlabel("Time (s); the red line marks the true onset")
fig.tight_layout()
fig.savefig(Path(FIGURE_DIR) / "causal_vs_zero_phase_demo.png", dpi=150)

Now the P wave at the nearest station, with a threshold at 10 times the noise (times after the origin from
`tr.times(reftime=ORIGIN)`).

**Watch out:** `detrend`, `taper` and `filter` change the trace **in place**. Work on `tr.copy()` when you still
need the original.

In [ ]:
demo = z_raw.copy()
kept = demo.data.copy()
returned = demo.detrend("demean")
print("detrend returned the same object:", returned is demo, "| its data changed:", not np.array_equal(kept, demo.data))

prepared = z_raw.copy().detrend("demean").taper(max_percentage=0.01, type="hann")
fig, ax = plt.subplots(figsize=(7.5, 3.2))
for name, zero_phase, color in (("causal", False, "tab:blue"), ("zero-phase", True, "tab:red")):
    version = prepared.copy().filter("bandpass", freqmin=FMIN_HZ, freqmax=FMAX_HZ, corners=4, zerophase=zero_phase)
    t_rel = version.times(reftime=ORIGIN)
    quiet = (t_rel > -25.0) & (t_rel < -5.0)
    noise = np.sqrt(np.mean(version.data[quiet] ** 2))
    first = t_rel[np.argmax((np.abs(version.data) > 10.0 * noise) & (t_rel > -5.0))]
    print(f"{name:>10s}: noise level {noise:7.1f} counts; first above 10 x noise at {first:5.2f} s after the origin")
    shown = (t_rel > 0.0) & (t_rel < 6.0)
    ax.plot(t_rel[shown], version.data[shown], color=color, lw=0.7, label=name)
    ax.axvline(first, color=color, lw=0.8, ls="--")
ax.set_xlabel("Time after origin (s)")
ax.set_ylabel(f"{z_raw.stats.channel} (counts)")
fig.legend(loc="upper center", ncol=2, frameon=False, fontsize=8)
fig.tight_layout(rect=(0, 0, 1, 0.88))
fig.savefig(Path(FIGURE_DIR) / "causal_vs_zero_phase_onset.png", dpi=150)

The zero-phase version crosses the threshold at 1.72 s, 2 s before the causal one (3.71 s).

**Watch out:** pick onsets and polarities on causal or unfiltered records; measure peaks and compare waveforms on
zero-phase records.

## Part 5 · Amplitude spectra (about 30 min)

The **Fourier transform** writes a record as a sum of sines; its **spectrum** tells how large each frequency is.
For N samples `delta` seconds apart, `np.fft.rfft` returns N // 2 + 1 numbers $X_k$. They belong to the
frequencies $k / (N \cdot \delta)$, from 0 Hz to the Nyquist frequency. The **amplitude spectrum** $2|X_k|/N$ gives
a sine of amplitude A a peak of height A. In **decibels**, 20 dB is a factor of 10.

### M10-T2 · Spectrum of a filtered trace · core · about 75 min

Write `band_amplitude_spectrum(tr, fmin, fmax)`. On a copy of `tr`, remove the mean, taper 1 % at each end (Hann)
and band-pass from `fmin` to `fmax` (Butterworth, `corners=4`, zero-phase). It returns `(freqs, amps)`: the
frequencies of `np.fft.rfft` and $2|X_k|/N$.

Example: 1000 samples at 100 Hz give 501 frequencies, 0, 0.1, ..., 50 Hz. A sine of amplitude 2 at 5 Hz, inside a
1-10 Hz band, gives `amps` close to 2 at 5 Hz.

The prediction and the Explain of this task are about the colleague's spectrum job, below.

**The colleague's job.** NumPy passes matrix products to a **BLAS** library, which can run them as **threads** on
several CPU cores. On a shared server, a job should use one. The colleague starts a spectrum job as a new Python
process (a **subprocess**) in three ways, A, B and C (next cell). A run's **CPU time** (all threads added up)
divided by its wall-clock time is the number of cores it kept busy.

In [ ]:
# The colleague's spectrum job. Run this cell (it writes the job to a file and runs nothing); do not edit it.
# Each run starts a new Python process with a clean environment in which the thread variables are set (without
# them the library would start one thread per core of the server), computes spectra for 5 s and prints how many
# cores it kept busy.
#   run A: starts with the thread variables at 4; the job sets them to 1 after it has imported NumPy
#   run B: starts with the thread variables at 1
#   run C: starts with the thread variables at 4; the job calls threadpoolctl's threadpool_limits(limits=1)
job_text = textwrap.dedent('''
    import os, sys, time
    import numpy as np                            # NumPy and its BLAS library are loaded here
    from threadpoolctl import threadpool_info, threadpool_limits

    n_samples, n_traces = 2048, 400               # the discrete Fourier transform written as a matrix
    k = np.arange(n_samples)
    dft = np.exp(-2j * np.pi * np.outer(k, k) / n_samples)
    traces = np.random.default_rng(1).standard_normal((n_samples, n_traces))
    test = np.abs(dft[:8, :8] @ traces[:8, :2])   # a quick test on a few samples first

    run = sys.argv[1]
    if run == "A":                                # the thread variables set to 1 from inside the job
        for name in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
            os.environ[name] = "1"
    limiter = threadpool_limits(limits=1) if run == "C" else None

    wall0, cpu0, rounds = time.perf_counter(), time.process_time(), 0
    while time.perf_counter() - wall0 < 5.0:
        spectra = np.abs(dft @ traces)
        rounds += 1
    wall, cpu = time.perf_counter() - wall0, time.process_time() - cpu0
    blas = [p["num_threads"] for p in threadpool_info() if p["user_api"] == "blas"]
    print(f"run {run}: {cpu / wall:4.2f} cores busy, {rounds * n_traces / wall:6.0f} spectra per second, "
          f"BLAS reports {blas} thread(s)")
''')
job_file = Path(RESULT_DIR) / "colleague_spectra_job.py"
job_file.write_text(job_text, encoding="utf-8")
clean_env = {"PATH": os.environ.get("PATH", ""), "HOME": os.environ.get("HOME", ""), "NUMPY_MADVISE_HUGEPAGE": "0"}
four = {name: "4" for name in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS")}
one = {name: "1" for name in four}
RUNS = {"A": four, "B": one, "C": four}            # the thread variables each run starts with
nice_prefix = ["nice", "-n", "15"] if shutil.which("nice") else []
print(f"job written to {job_file}; runs A, B and C start with",
      ", ".join(f"{run}: {start['OMP_NUM_THREADS']}" for run, start in RUNS.items()), "threads allowed")

In [ ]:
# Before you code: how many cores do you expect runs A, B and C of the colleague's job to keep busy?
st.predict("M10-T2", """

""")

Now measure the three runs (about 20 s) and compare them with your prediction.

In [ ]:
for run_name, start_env in RUNS.items():
    run = subprocess.run(nice_prefix + [sys.executable, "-I", str(job_file), run_name],
                         env={**clean_env, **start_env}, capture_output=True, text=True, timeout=120)
    print(run.stdout.strip() or run.stderr.strip()[-500:])



In [ ]:
def band_amplitude_spectrum(tr, fmin, fmax):
    """One-sided amplitude spectrum of a trace after a zero-phase Butterworth band-pass (corners=4).

    tr         : obspy Trace, evenly sampled at any rate; the data may be whole numbers and have an offset.
                 It is not changed: work on a copy.
    fmin, fmax : corner frequencies in Hz, 0 < fmin < fmax < Nyquist
    returns    : (freqs, amps), two 1-D numpy arrays of the same length: freqs from 0 Hz to the Nyquist frequency
                 in steps of 1 / (npts * delta), the frequencies of np.fft.rfft; amps = 2 |X_k| / npts of the copy
                 after removing the mean, a 1 % Hann taper at each end and the band-pass, in the units of tr
    """
    # Step 1: prepare a copy: mean removed, ends tapered, band-pass applied (zero-phase, corners=4).
    # Step 2: the discrete Fourier transform of the real-valued samples, and the frequency of each value.
    # Step 3: the amplitude scale of Part 5.
    # YOUR CODE HERE
    raise NotImplementedError("M10-T2: write your code here")

In [ ]:
st.check("M10-T2", func=band_amplitude_spectrum)

Your function on the nearest station: noise before the earthquake against the first 20 s after the origin.

In [ ]:
z_noise = z_raw.slice(ORIGIN - 25.0, ORIGIN - 5.0)
z_signal = z_raw.slice(ORIGIN, ORIGIN + 20.0)
f_noise, a_noise = band_amplitude_spectrum(z_noise, 0.2, 45.0)
f_signal, a_signal = band_amplitude_spectrum(z_signal, 0.2, 45.0)
fig, ax = plt.subplots(figsize=(6.5, 4.0))
ax.loglog(f_signal[1:], a_signal[1:], color="k", lw=0.6, label="0-20 s after origin")
ax.loglog(f_noise[1:], a_noise[1:], color="0.6", lw=0.6, label="25-5 s before origin")
ax.set_xlim(0.1, 50.0)
ax.set_xlabel("Frequency (Hz)")
ax.set_ylabel(f"{z_raw.stats.channel} amplitude (counts)")
fig.legend(loc="upper center", ncol=2, frameon=False, fontsize=8)
fig.tight_layout(rect=(0, 0, 1, 0.9))
fig.savefig(Path(FIGURE_DIR) / "near_station_spectra.png", dpi=150)
for f_hz in (0.5, 1.0, 2.0, 5.0, 10.0, 20.0, 40.0):
    near_k = np.abs(f_signal - f_hz) <= 0.25 * f_hz        # average over a band around f_hz
    ratio = a_signal[near_k].mean() / a_noise[np.abs(f_noise - f_hz) <= 0.25 * f_hz].mean()
    print(f"{f_hz:5.1f} Hz: signal / noise = {ratio:8.1f} ({20.0 * np.log10(ratio):5.1f} dB)")

**Explain:** Why did run A keep more cores busy than runs B and C?

## Part 6 · Counts and ground motion: testing a response removal (about 60 min)

The responses come from **StationMetadata**, integrated by Byeong-Seok Ahn (Pukyong National University). Cite
Ahn, Kang and Jung (2024) whenever you use it. Here you test the response removal of M03.

In [ ]:
ack = META_DIR / "ACKNOWLEDGMENT.txt"
xml_dir = META_DIR / "stationxml"
if not xml_dir.is_dir():
    raise FileNotFoundError(f"No station metadata in {xml_dir}. Install it: bin/sgtl-install metadata --apply")
print("\n".join(ack.read_text(encoding="utf-8").splitlines()[:12]))

The curves show how many counts each sensor of the nearest station writes for 1 m/s of ground velocity. The
broadband seismometer measures velocity, so its curve is flat. The accelerometer measures acceleration, so its
curve rises with frequency.

In [ ]:
near_inv = read_inventory(str(xml_dir / f"{NEAR_STATION}.xml"))
freqs_hz = np.logspace(-3, np.log10(45.0), 300)
fig, ax = plt.subplots(figsize=(6.5, 4.0))
for code, color in ((f"{BAND}Z", "k"), ("HGZ", "tab:orange")):
    seed_id = f"{NEAR_STATION}..{code}"
    resp = near_inv.get_response(seed_id, ORIGIN)
    per_velocity = np.abs(resp.get_evalresp_response_for_frequencies(freqs_hz, output="VEL"))
    ax.loglog(freqs_hz, per_velocity, color=color, lw=1.0, label=code)
    print(f"{seed_id}: {per_velocity[np.argmin(np.abs(freqs_hz - 0.1))]:.4g} counts per m/s at 0.1 Hz, "
          f"{per_velocity[np.argmin(np.abs(freqs_hz - 1.0))]:.4g} at 1 Hz, "
          f"{per_velocity[np.argmin(np.abs(freqs_hz - 10.0))]:.4g} at 10 Hz")
ax.set_xlabel("Frequency (Hz)")
ax.set_ylabel("Counts per m/s of ground velocity")
fig.legend(loc="upper center", ncol=2, frameon=False, fontsize=8)
fig.tight_layout(rect=(0, 0, 1, 0.9))
fig.savefig(Path(FIGURE_DIR) / "near_station_response.png", dpi=150)

The cell removes the response as in M03, with the **pre-filter** as the safeguard where the response is tiny.
ObsPy's other safeguard, a **water level** (a floor under the response), is switched off: use one, and say which.
`output="VEL"` gives ground velocity in m/s.

In [ ]:
vel = near.copy().detrend("demean").taper(max_percentage=0.05, type="hann")
vel.remove_response(inventory=near_inv, output="VEL", pre_filt=PRE_FILT_HZ, water_level=None)
fig, axes = plt.subplots(3, 1, figsize=(7.5, 5.2), sharex=True)
for ax, counts_tr, vel_tr in zip(axes, near, vel):
    t_rel = vel_tr.times(reftime=ORIGIN)
    ax.plot(t_rel, vel_tr.data * 1000.0, color="k", lw=0.4)
    ax.set_ylabel(f"{vel_tr.stats.channel} (mm/s)")
    print(f"{vel_tr.id}: largest |counts - mean| {np.abs(counts_tr.data - counts_tr.data.mean()).max():10.0f} -> "
          f"largest |velocity| {np.abs(vel_tr.data).max() * 1000.0:.3f} mm/s")
axes[-1].set_xlim(-5.0, 40.0)
axes[-1].set_xlabel("Time after origin (s)")
fig.tight_layout()
fig.savefig(Path(FIGURE_DIR) / "near_station_velocity.png", dpi=150)

**A test of the result.** Most stations have both a broadband seismometer (HH) and an accelerometer (HG). With
correct responses, both show the same ground velocity. The cell compares them at the nearest stations.

In [ ]:
def velocity_record(path, inventory):
    """Ground velocity (m/s) of one SAC file: detrend, taper, response removal, band-pass, cut."""
    trace = read(str(path))[0]
    trace.detrend("demean").taper(max_percentage=0.05, type="hann")
    trace.remove_response(inventory=inventory, output="VEL", pre_filt=PRE_FILT_HZ, water_level=None)
    trace.filter("bandpass", freqmin=FMIN_HZ, freqmax=FMAX_HZ, corners=4, zerophase=True)
    return trace.trim(ORIGIN - 5.0, ORIGIN + 40.0)


def station_distance_km(path):
    """Epicentral distance (km) from the SAC header's station coordinates to the KMA epicenter."""
    sac = read(str(path), headonly=True)[0].stats.sac
    return gps2dist_azimuth(EV_LAT, EV_LON, sac.stla, sac.stlo)[0] / 1000.0


hh_z_files = sorted((SAC_DIR / "HH").glob("*HHZ.sac"))
by_distance = sorted(hh_z_files, key=station_distance_km)
rows = []
for z_file in by_distance:
    net_sta = ".".join(z_file.name.split(".")[1:3])
    if len(rows) >= 3 * N_COLOCATED or not list((SAC_DIR / "HG").glob(f"*.{net_sta}.HGZ.sac")):
        continue
    inv = read_inventory(str(xml_dir / f"{net_sta}.xml"))
    for comp in "ZNE":
        hh = velocity_record(next((SAC_DIR / "HH").glob(f"*.{net_sta}.HH{comp}.sac")), inv)
        hg = velocity_record(next((SAC_DIR / "HG").glob(f"*.{net_sta}.HG{comp}.sac")), inv)
        n = min(hh.stats.npts, hg.stats.npts)
        depth = {code: inv.select(channel=code, time=ORIGIN)[0][0][0].depth for code in ("HH" + comp, "HG" + comp)}
        rows.append({"station": net_sta, "component": comp, "epi_km": station_distance_km(z_file),
                     "HH_depth_m": depth["HH" + comp], "HG_depth_m": depth["HG" + comp],
                     "HH_peak_mm_s": 1000.0 * np.abs(hh.data).max(), "HG_peak_mm_s": 1000.0 * np.abs(hg.data).max(),
                     "HH/HG": np.abs(hh.data).max() / np.abs(hg.data).max(),
                     "correlation": np.corrcoef(hh.data[:n], hg.data[:n])[0, 1]})
colocated = pd.DataFrame(rows)
print(colocated.round({"epi_km": 1, "HH_peak_mm_s": 3, "HG_peak_mm_s": 3, "HH/HG": 2, "correlation": 3})
      .to_string(index=False))

At the same depth, the two agree within a few percent, with a correlation near 1, as at KS.CHJ2. Elsewhere ask
why: different depths (some sensors sit in 100 m boreholes), a wrong gain or orientation. A correlation near −1
means a **polarity reversal** that the metadata do not describe.

**Radial and transverse:** `rotate("NE->RT", back_azimuth=...)` turns the horizontals toward the earthquake
(**radial**) and across (**transverse**). The **back azimuth** is the direction from the station to the
earthquake.

In [ ]:
dist_m, azimuth_deg, back_azimuth_deg = gps2dist_azimuth(EV_LAT, EV_LON, z_raw.stats.sac.stla, z_raw.stats.sac.stlo)
rt = vel.copy().filter("bandpass", freqmin=FMIN_HZ, freqmax=FMAX_HZ, corners=4, zerophase=True)
rt.rotate("NE->RT", back_azimuth=back_azimuth_deg)
print(f"{NEAR_STATION}: epicentral distance {dist_m / 1000.0:.1f} km, back azimuth {back_azimuth_deg:.1f} degrees")
for trace in rt.sort(keys=["channel"]):
    print(f"{trace.id}: largest |velocity| {np.abs(trace.data).max() * 1000.0:.3f} mm/s")

*Optional:* **explorer E1**, a filter bench with your `band_amplitude_spectrum`. Try causal against zero-phase.

In [ ]:
# Optional explorer E1 (not checked): the vertical record of the nearest station on the filter bench.
st.widgets.filter_bench(z_raw, spectrum=band_amplitude_spectrum, inventory=near_inv, pre_filt=PRE_FILT_HZ,
                        fmin=FMIN_HZ, fmax=FMAX_HZ, reftime=ORIGIN, window=(-5.0, 40.0),
                        xlabel="Time after origin (s)")

## Part 7 · Distance from the S-P time (about 20 min)

The **P wave** is faster than the **S wave**, so the delay between them, the **S-P time**, grows with distance. On
a straight path of length $R$ at constant speeds $V_P$ and $V_S$, the travel times are $R/V_P$ and $R/V_S$. Here $R$
is the **hypocentral distance**: $R^2 = \Delta^2 + h^2$, with $\Delta$ the epicentral distance and $h$ the depth.

### M10-T3 · Distance from an S-P time · core · about 30 min

Write `hypo_distance_from_sp(sp_s, vp, vs)`. It returns the hypocentral distance in km that gives the S-P time
`sp_s` (s) for the speeds `vp` and `vs` (km/s). `sp_s` can be a number or a numpy array.

Example: `vp` = 6.0, `vs` = 3.5 and 1.0 s give 8.4 km.

In [ ]:
# Before you code: with the top layer of kim1983 (VP 5.98, VS 3.40 km/s), how many kilometers does each second
# of S-P time stand for?
st.predict("M10-T3", """

""")

In [ ]:
def hypo_distance_from_sp(sp_s, vp, vs):
    """Hypocentral distance (km) from an S-P time, for straight paths at constant speeds.

    sp_s   : S-P time in seconds, a number or a 1-D numpy array
    vp, vs : P and S speeds in km/s (vp > vs)
    returns: distance in km: a float for a number, a numpy array of the same shape for an array
    """
    # Step 1: write the S-P time as the difference of the two travel times of one distance.
    # Step 2: solve that for the distance, for every value at once.
    # YOUR CODE HERE
    raise NotImplementedError("M10-T3: write your code here")

In [ ]:
st.check("M10-T3", func=hypo_distance_from_sp)

Real S-P times: ObsPy's automatic picker `ar_pick` gives rough P and S times at the `N_PICKED` nearest stations
(M20 picks properly). Your S-P distances against the catalog distances:

In [ ]:
from obspy.signal.trigger import ar_pick

picked = []
for z_file in by_distance[:N_PICKED]:
    net_sta = ".".join(z_file.name.split(".")[1:3])
    three = read(str(SAC_DIR / BAND / f"*.{net_sta}.{BAND}?.sac")).trim(ORIGIN - 20.0, ORIGIN + 40.0)
    three.detrend("demean")
    z, n, e = (three.select(component=c)[0] for c in "ZNE")
    p_s, s_s = ar_pick(z.data, n.data, e.data, z.stats.sampling_rate, 1.0, 20.0, 1.0, 0.1, 4.0, 1.0, 2, 8, 0.1, 0.2)
    epi_km = station_distance_km(z_file)
    hypo_km = float(np.hypot(epi_km, EV_DEPTH_KM))
    sp_km = hypo_distance_from_sp(s_s - p_s, VP_KM_S, VS_KM_S)
    picked.append({"station": net_sta, "P_after_origin_s": z.stats.starttime + p_s - ORIGIN,
                   "S_after_origin_s": z.stats.starttime + s_s - ORIGIN, "S-P_s": s_s - p_s,
                   "hypo_km": hypo_km, "from_S-P_km": sp_km, "difference_%": 100.0 * (sp_km - hypo_km) / hypo_km})
picked = pd.DataFrame(picked)
print(picked.round(2).to_string(index=False))
print(f"median difference {picked['difference_%'].median():.1f} %")

**Explain:** Are your S-P distances longer or shorter than the catalog distances, and what could cause that?

## Part 8 · Record sections (about 25 min)

A **record section** draws many seismograms, each at its distance from the source, against time after the origin.
A wave that travels straight at a constant speed v lines up on t = R / v. House rule for every figure: **print every
number your figure shows**.

### M10-T4 · A record section · core · about 90 min

Write `record_section(stream_or_paths, origin, event_lat, event_lon, event_depth_km, fmin=1.0, fmax=10.0,
scale_km=2.0)`. It filters copies of the traces as in task 2 and draws each one as a line at its hypocentral
distance. It returns `(fig, table)` and prints the table. The docstring lists the axes, labels and columns.

Use `gps2dist_azimuth` for distances. Not allowed: pyproj.

In [ ]:
# Before you code: how far apart in time do you expect the P and the S arrivals at 50 km and at 100 km?
st.predict("M10-T4", """

""")

In [ ]:
def record_section(stream_or_paths, origin, event_lat, event_lon, event_depth_km, fmin=1.0, fmax=10.0,
                   scale_km=2.0):
    """Record section: each trace at its hypocentral distance, against time after the origin.

    stream_or_paths : obspy Stream (not changed), or a list of SAC paths (str or Path); station coordinates in
                      tr.stats.sac.stla and tr.stats.sac.stlo (degrees)
    origin          : UTCDateTime of the origin
    event_lat, event_lon, event_depth_km : hypocenter (degrees, degrees, km)
    fmin, fmax      : band-pass corner frequencies (Hz)
    scale_km        : half-height of each trace on the distance axis (km)
    Processing, on copies, as in task 2: remove the mean, taper 1 % (Hann), band-pass zero-phase (corners=4).
    epi_km = epicentral distance (km); hypo_km = sqrt(epi_km**2 + event_depth_km**2); peak = largest absolute
    value of the filtered trace.
    returns         : (fig, table)
        fig   : one Axes; each trace is ONE ax.plot line with all its samples: x = seconds after the origin
                (negative before it), y = hypo_km + scale_km * trace / peak. x label: time after origin (s);
                y label: distance (km); each trace's station code next to it (ax.text).
        table : DataFrame with the columns id (tr.id), epi_km, hypo_km, peak; one row per trace, sorted by
                hypo_km (closest first). Print the whole table.
    """
    # Step 1: one Stream of copies, whatever the input was.
    # Step 2: for each trace: filter, distances, peak.
    # Step 3: sort, draw one line per trace with labels, build and print the table.
    # YOUR CODE HERE
    raise NotImplementedError("M10-T4: write your code here")

In [ ]:
st.check("M10-T4", func=record_section, dataset=DATASET)

Your function on all `BAND` Z records, with the P line (red) and the S line (blue) of the `kim1983` top layer. Each
trace has its own scale: compare amplitudes with the `peak` column.

In [ ]:
sangju_fig, sangju_table = record_section(sorted((SAC_DIR / BAND).glob(f"*{BAND}Z.sac")), ORIGIN, EV_LAT, EV_LON,
                                          EV_DEPTH_KM, fmin=FMIN_HZ, fmax=FMAX_HZ, scale_km=SCALE_KM)
ax = sangju_fig.axes[0]
r_line = np.array([0.0, sangju_table["hypo_km"].max() + 5.0])
for speed, color, name in ((VP_KM_S, "tab:red", "P"), (VS_KM_S, "tab:blue", "S")):
    ax.plot(r_line / speed, r_line, color=color, lw=1.0, ls="--", label=f"{name}: {speed:g} km/s")
    print(f"{name} line: t = R / {speed} km/s")
# Provided layout: separate labels without moving or excluding any seismic trace.
labels = sorted(ax.texts, key=lambda text: text.get_position()[1])
distances = sangju_table["hypo_km"].to_numpy()
spacing = (distances.max() - distances.min()) / (len(distances) + 3)
levels = distances.copy()
for i in range(1, len(levels)):
    levels[i] = max(levels[i], levels[i - 1] + spacing)
levels -= (levels - distances).mean()
for text, distance, level, trace_id in zip(labels, distances, levels, sangju_table["id"]):
    text.set_position((1.045, level))
    text.set_text(".".join(trace_id.split(".")[:2]))
    text.set_fontsize(8)
    ax.plot([1.002, 1.035], [distance, level], color="0.6", lw=0.5,
            transform=ax.get_yaxis_transform(), clip_on=False)
ax.set_ylim(min(distances.min(), levels.min()) - 4, max(distances.max(), levels.max()) + 4)
sangju_fig.set_size_inches(9, 10)
sangju_fig.subplots_adjust(right=0.80)
ax.legend(loc="upper left", fontsize=9)
ax.set_xlim(-5.0, 45.0)
ax.set_title(f"Sangju ML {EV_ML:.1f}, {BAND}Z, {FMIN_HZ:g}-{FMAX_HZ:g} Hz", fontsize=10)
sangju_fig.savefig(Path(FIGURE_DIR) / "sangju_record_section.png", dpi=150)
print(f"{len(sangju_table)} traces, {sangju_table['hypo_km'].min():.1f} to {sangju_table['hypo_km'].max():.1f} km; "
      f"x from -5 to 45 s after {ORIGIN}; each trace scaled to +-{SCALE_KM} km")

**Explain:** Do the P and S arrivals fall on their two lines, and where do they depart from them?

## Stretch · How strong was the shaking?

**Peak ground velocity** (PGV) is the largest absolute ground velocity of a record. Is yours reasonable? M03's
magnitude formula, run backward, gives the largest Wood-Anderson amplitude $A$ expected at the distance $R$:

$$\log_{10}(A / \mathrm{nm}) = M_\mathrm{L} - 1.11 \log_{10}(R / \mathrm{km}) - 0.00189\, (R / \mathrm{km}) + 2.09$$

A swing of frequency $f$ has a peak velocity of about $2\pi f A$. The formula is Californian, so expect agreement
within a factor of a few.

### M10-T5 · Peak ground velocity against a magnitude expectation · stretch · about 100 min

Compare the PGV of `PGV_TRACE` with the expectation above:

1. remove the response as in Part 6 and measure the PGV;
2. find `hypo_km` and the spectral peak `f_peak_hz` (your `band_amplitude_spectrum`);
3. compute the expectation, write a JSON file and save a two-panel figure;
4. set `pgv_result`.

The comments in the code cell list every key and panel.

In [ ]:
# Before you code: will the measured PGV be larger or smaller than the expectation, and by about what factor?
st.predict("M10-T5", """

""")

In [ ]:
# Remove the response of PGV_TRACE, measure PGV, distance and spectral peak, compute the expectation, write the
# JSON file, draw and save the two-panel figure, and set pgv_result:
# - read the SAC file of PGV_TRACE in SAC_DIR and its station's StationXML in META_DIR / "stationxml". On a copy,
#   remove the mean, taper 5 % (Hann) and remove_response(output="VEL", pre_filt=PRE_FILT_HZ, water_level=None),
#   with no further filter. pgv_m_s = the largest absolute value of the whole record, in m/s.
# - hypo_km: from the station coordinates in the SAC header and the KMA hypocenter (EV_LAT, EV_LON, EV_DEPTH_KM).
# - f_peak_hz: the frequency of the largest value of your band_amplitude_spectrum of the whole velocity record,
#   with the corner frequencies FPEAK_BAND_HZ.
# - expected_pgv_m_s = 2 pi f_peak_hz A, with A from the formula above (nm converted to m);
#   ratio = pgv_m_s / expected_pgv_m_s.
# - the JSON file RESULT_DIR/pgv_check.json with sac_path and stationxml (paths as text), event_id (KMA_EVENT_ID),
#   ml, hypo_km, pgv_m_s, f_peak_hz, expected_pgv_m_s and ratio (numbers).
# - one figure with two panels, saved as a PNG file: (1) the velocity from 5 s before to 40 s after the origin,
#   seconds after the origin on the x axis, with a marker at the PGV; (2) your spectrum inside FPEAK_BAND_HZ with
#   a marker at f_peak_hz. Each y label names its unit (m/s or mm/s). Write pgv_m_s, f_peak_hz, expected_pgv_m_s
#   and ratio in the figure (title, text or legend), with at least two significant digits.
# - pgv_result = {"json": <JSON path>, "png": <PNG path>, "figure": <the Figure>,
#                 "spectrum": band_amplitude_spectrum}
# YOUR CODE HERE
raise NotImplementedError("M10-T5: write your code here")

In [ ]:
st.check("M10-T5", answer=pgv_result)

**Explain:** By what factor does your PGV differ from the expectation, and what is the most likely reason?

## Wrap-up

Your progress in this module (from `work/.sgtl/progress.jsonl`):

In [ ]:
st.progress("M10")

### Going further (optional, not checked)

1. Repeat Part 7 with a Vp/Vs ratio of 1.70 and of 1.80. Which fits the catalog distances best?

**Read more:** Helffrich, Wookey and Bastow (2013), SAC; Scherbaum (2001), filters; Havskov and Ottemöller
(2010), data processing.

Next: M20 picks P and S arrivals on these same records.